In [ ]:
# Setup
import importlib
import os
import subprocess
import sys
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
REPO_PATH = Path('/content/AI_Agentic_DL') if IS_COLAB else Path.cwd()
TARGET_BRANCH = 'final-pipeline-integration'
REQUIRED_DATA_FILES = ['X_train.npy', 'X_test.npy', 'y_train.npy', 'y_test.npy']
ENV_MARKER = '/tmp/final_pipeline_env_ready_v3'

if IS_COLAB and not REPO_PATH.exists():
    !git clone https://github.com/Lawapaul/AI_Agentic_DL.git /content/AI_Agentic_DL

if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

subprocess.check_call(['git', '-C', str(REPO_PATH), 'fetch', 'origin'])
subprocess.check_call(['git', '-C', str(REPO_PATH), 'checkout', TARGET_BRANCH])
subprocess.check_call(['git', '-C', str(REPO_PATH), 'pull', 'origin', TARGET_BRANCH])

def ensure_package(requirement, import_name=None):
    import_name = import_name or requirement.split('==')[0].split('>=')[0].replace('-', '_')
    try:
        importlib.import_module(import_name)
        return False
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', requirement])
        return True

if not os.path.exists(ENV_MARKER):
    changed = False
    changed |= ensure_package('lime>=0.2.0.1', 'lime')
    changed |= ensure_package('transformers==4.41.2', 'transformers')
    changed |= ensure_package('accelerate==0.30.1', 'accelerate')
    changed |= ensure_package('sentencepiece>=0.1.99', 'sentencepiece')
    changed |= ensure_package('networkx>=3.0.0', 'networkx')
    changed |= ensure_package('pyarrow>=15.0.0', 'pyarrow')
    Path(ENV_MARKER).write_text('ready', encoding='utf-8')
    if changed:
        raise SystemExit('Installed missing packages. Restart runtime once, then rerun this cell.')

for module_name in list(sys.modules):
    if module_name.startswith(('transformers', 'accelerate', 'tokenizers', 'huggingface_hub')):
        del sys.modules[module_name]

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

DEFAULT_PROCESSED_PATHS = [
    '/content/drive/MyDrive/Deep Learning Project/AI Agentic/data/processed',
    '/content/drive/MyDrive/AI Agentic/data/processed',
    '/content/AI_Agentic_DL/data/processed',
    str(REPO_PATH / 'data' / 'processed'),
]
DEFAULT_MODEL_PATHS = [
    '/content/drive/MyDrive/Deep Learning Project/AI Agentic/saved_models/hybrid_cnn_lstm_full_2_2m.keras',
    '/content/drive/MyDrive/Deep Learning Project/AI Agentic/saved_models/ids_model.keras',
    '/content/drive/MyDrive/AI Agentic/saved_models/hybrid_cnn_lstm_full_2_2m.keras',
    '/content/AI_Agentic_DL/saved_models/ids_model.keras',
    str(REPO_PATH / 'saved_models' / 'ids_model.keras'),
]

def first_existing_processed(paths):
    for path in paths:
        path_obj = Path(path)
        if path_obj.exists() and all((path_obj / name).exists() for name in REQUIRED_DATA_FILES):
            return str(path_obj)
    return None

def first_existing_file(paths):
    for path in paths:
        if Path(path).exists():
            return path
    return None

PROCESSED_PATH = first_existing_processed(DEFAULT_PROCESSED_PATHS)
MODEL_PATH = first_existing_file(DEFAULT_MODEL_PATHS)

if PROCESSED_PATH is None:
    raise FileNotFoundError('Could not find processed dataset files. Checked: ' + ', '.join(DEFAULT_PROCESSED_PATHS))

if MODEL_PATH is None:
    raise FileNotFoundError('Could not find a trained hybrid model. Checked: ' + ', '.join(DEFAULT_MODEL_PATHS))

print('Repo branch:', TARGET_BRANCH)
print('Processed path:', PROCESSED_PATH)
print('Model path:', MODEL_PATH)


In [ ]:
# Load modules
import importlib
import sys

importlib.invalidate_caches()
if 'integration.final_pipeline' in sys.modules:
    del sys.modules['integration.final_pipeline']

import integration.final_pipeline as final_pipeline
final_pipeline = importlib.reload(final_pipeline)
run_demo_pipeline = final_pipeline.run_demo_pipeline
run_full_pipeline = final_pipeline.run_full_pipeline


In [ ]:
# Run full pipeline (safe mode)
full_summary = run_full_pipeline(PROCESSED_PATH, MODEL_PATH)
print(full_summary['status'])
print(full_summary['runtime'])
print(full_summary['decision_planner'])
print(full_summary['retraining'])


In [ ]:
# Run demo pipeline (presentation mode)
demo_summary = run_demo_pipeline()
print('Demo samples:', len(demo_summary.get('records', [])))
